# List.Buffer i Table.Buffer w Power Query — od podstaw do mistrzostwa

**Uwaga techniczna:** komórki `code` w tym notebooku zawierają kod w języku **M (Power Query)**, nie Python.
Ten plik nie jest przeznaczony do uruchamiania w Jupyterze — to referencyjny przewodnik do wklejania kodu wprost do edytora zapytań Power Query (Power BI Desktop / Excel) lub do Advanced Editor. Podświetlanie składni będzie "pythonowe" (kosmetyczne), ale to nie ma znaczenia funkcjonalnego.

**Spis treści:**
1. Zanim zrozumiesz Buffer — musisz zrozumieć lazy evaluation
2. Analogia "jak dla dziecka"
3. `List.Buffer` — składnia i pierwszy przykład
4. `Table.Buffer` — składnia i pierwszy przykład
5. Różnice między `List.Buffer` a `Table.Buffer`
6. Klasyczna pułapka wydajnościowa: `List.Generate` bez buforowania
7. Zaawansowane triki produkcyjne
8. Buffer a query folding — najważniejszy kompromis w całym temacie
9. Kiedy **NIE** stosować Buffer — checklist
10. Jak zmierzyć efekt buforowania (Query Diagnostics)
11. Cheatsheet / ściąga decyzyjna
12. Ćwiczenia do samodzielnego przetestowania


## 1. Zanim zrozumiesz Buffer — musisz zrozumieć lazy evaluation

Power Query (silnik M) jest **leniwy** (*lazy*). To fundament całego tematu, więc zatrzymajmy się tu na chwilę, bo stąd biorą się wszystkie problemy, które rozwiązują `List.Buffer` i `Table.Buffer`.

### Co to znaczy "leniwy silnik"?

W typowym języku (Python, SQL wykonywany krok po kroku) instrukcja jest wykonywana w momencie, gdy ją napiszesz/uruchomisz. W M jest inaczej:

- Krok w Power Query (`let ... in`) **nie jest wykonywany w momencie zdefiniowania**.
- Krok jest wykonywany **dopiero wtedy, gdy jego wynik jest faktycznie potrzebny** (np. do wyświetlenia podglądu, do załadowania tabeli, do obliczenia kolejnego kroku).
- Co gorsza: **jeśli ten sam krok jest referencjonowany wielokrotnie**, silnik M **może przeliczyć go od nowa za każdym razem**, zamiast trzymać wynik w pamięci.

To jest kluczowe zdanie tego całego dokumentu: **odwołanie się do zmiennej w M nie gwarantuje odczytania gotowego wyniku — może oznaczać ponowne przeliczenie całej gałęzi zapytania od źródła.**

### Dlaczego tak to zaprojektowano?

Bo silnik M stara się być maksymalnie **inteligentny i oszczędny**: jeśli nie musi czegoś liczyć (bo np. dana kolumna nigdy nie trafia do wyniku), to tego nie liczy. To świetne dla wydajności w 90% przypadków — ale w pozostałych 10% (pętle, wielokrotne odwołania, funkcje rekurencyjne, źródła "niestabilne") staje się **pułapką wydajnościową rzędu O(n²) lub gorzej**.

`List.Buffer` i `Table.Buffer` to Twój sposób powiedzenia silnikowi:

> "Oblicz to raz, zapisz w pamięci RAM jako statyczną, zamrożoną kopię i **nie licz tego drugi raz**."


## 2. Analogia "jak dla dziecka"

Wyobraź sobie, że pracujesz w bibliotece (to jest Twoje źródło danych — np. SQL Server, plik Excel, API).

Masz asystenta (silnik Power Query), który jest **bardzo dokładny, ale strasznie leniwy w pamiętaniu rzeczy**. Za każdym razem, gdy o coś go zapytasz — mówi: *"Zaraz, muszę pójść na półkę i sprawdzić"* — i idzie po książkę od nowa. Nawet jeśli pytałeś go o to samo 5 minut temu.

- Zapytasz go raz o książkę → idzie, przynosi, odpowiada. OK, to normalne.
- Ale jeśli w pętli (np. dla każdego wiersza tabeli, 50 000 razy) zadajesz mu **to samo pytanie o tę samą listę/tabelę** → **on 50 000 razy idzie na tę samą półkę**, zamiast raz przynieść książkę i położyć ją na biurku przed sobą.

`List.Buffer(lista)` / `Table.Buffer(tabela)` to jest dokładnie to:

> **"Asystencie, idź na tę półkę TYLKO RAZ, połóż tę książkę na biurku (w pamięci RAM) i od teraz odpowiadaj z pamięci, patrząc na biurko, a nie chodząc na półkę."**

Efekt: zamiast tysięcy podróży na półkę (przeliczeń od źródła), masz jedną podróż i tysiące szybkich spojrzeń na biurko.

**Ale uwaga (to jest kluczowe i wraca w sekcji 8):** jeśli ta "półka" to serwer SQL, który potrafił sam, po swojej stronie, filtrować i agregować dane szybciej niż Twój komputer (tzw. **query folding**) — to każąc asystentowi "przynieść książkę na biurko" **zabierasz mu możliwość dalszej pracy na półce**. Wszystko, co zrobisz *po* Buffer, będzie liczone lokalnie, na Twoim laptopie, a nie na serwerze. Czasem to dobrze, czasem to bardzo źle.


## 3. `List.Buffer` — składnia i pierwszy przykład

### Składnia

```
List.Buffer(list as list) as list
```

| Parametr | Wymagany? | Opis |
|---|---|---|
| `list` | **wymagany** | dowolna lista M, np. wynik `Table.ToList`, `List.Generate`, `{1..100}` |

Zwraca: **tę samą listę**, ale w postaci w pełni zmaterializowanej w pamięci — "zamrożoną" migawkę wartości i kolejności w momencie buforowania.

### Prosty przykład — bez sensu praktycznego, ale pokazuje mechanikę


In [ ]:
// Bez buforowania — każde odwołanie do MojaLista
// teoretycznie może wywołać ponowne przeliczenie źródła
let
    MojaLista = List.Numbers(1, 5),
    Suma = List.Sum(MojaLista),
    Srednia = List.Average(MojaLista),
    Max = List.Max(MojaLista)
in
    [Suma = Suma, Srednia = Srednia, Max = Max]

// Z buforowaniem — MojaLista jest liczona RAZ,
// a Suma / Srednia / Max czytają już gotową kopię z pamięci
let
    MojaLista = List.Buffer(List.Numbers(1, 5)),
    Suma = List.Sum(MojaLista),
    Srednia = List.Average(MojaLista),
    Max = List.Max(MojaLista)
in
    [Suma = Suma, Srednia = Srednia, Max = Max]

**Uczciwie:** dla listy 5-elementowej z `List.Numbers` różnicy praktycznie nie zobaczysz — silnik i tak by to zoptymalizował lub koszt jest znikomy. Buffer pokazuje swoją wartość dopiero przy **dużych listach, referencjonowanych wielokrotnie, zwłaszcza w kontekście pętli/rekurencji** (patrz sekcja 6) albo przy **niestabilnych źródłach** (patrz sekcja 7).


## 4. `Table.Buffer` — składnia i pierwszy przykład

### Składnia

```
Table.Buffer(table as table) as table
```

| Parametr | Wymagany? | Opis |
|---|---|---|
| `table` | **wymagany** | dowolna tabela M |

Zwraca: tabelę zmaterializowaną w pamięci — **zamrożone dane, kolejność wierszy i typy kolumn** w momencie buforowania.

### Prosty przykład


In [ ]:
let
    Zrodlo = Sql.Database("SerwerX", "BazaY"),
    Tabela = Zrodlo{[Schema="dbo", Item="Sprzedaz"]}[Data],
    // Buforujemy PO wykonaniu filtrów, które mogą się foldować do SQL
    Przefiltrowane = Table.SelectRows(Tabela, each [Rok] = 2025),
    Zbuforowane = Table.Buffer(Przefiltrowane),

    // Poniższe kroki NIE odpytują już serwera SQL wielokrotnie —
    // czytają z pamięci lokalnej
    KolumnaA = Table.AddColumn(Zbuforowane, "SumaWiersza", each [Ilosc] * [Cena]),
    KolumnaB = Table.AddColumn(KolumnaA, "Ranking", each
        List.PositionOf(
            List.Sort(Table.Column(Zbuforowane, "Ilosc"), Order.Descending),
            [Ilosc]
        ) + 1
    )
in
    KolumnaB

Zwróć uwagę na wzorzec `KolumnaB`: kolumna `Ranking` odwołuje się do **całej kolumny `Zbuforowane`** dla **każdego wiersza** (klasyczny anti-pattern rangowania w M, o którym więcej w sekcji 7.3). Bez `Table.Buffer` w tym miejscu — przy większej tabeli — to jest przepis na katastrofę wydajnościową, bo `Table.Column(Zbuforowane, "Ilosc")` mogłoby być przeliczane od nowa dla każdego wiersza.


## 5. Różnice między `List.Buffer` a `Table.Buffer`

| Cecha | `List.Buffer` | `Table.Buffer` |
|---|---|---|
| Typ wejścia/wyjścia | `list` → `list` | `table` → `table` |
| Co "zamraża" | wartości i kolejność elementów listy | wartości, kolejność wierszy **i typy kolumn** |
| Typowe zastosowanie | akumulatory w `List.Generate`, listy używane wielokrotnie w obliczeniach | tabele używane w wielu joinach/self-referencjach, stabilizacja przed indeksowaniem |
| Wpływ na query folding | przerywa folding od tego miejsca w dół | przerywa folding od tego miejsca w dół |
| Koszt pamięciowy | cała lista w RAM | cała tabela w RAM (wszystkie zbuforowane kolumny × wiersze) |
| Częstość użycia w praktyce | rzadziej, głównie w zaawansowanej logice list/rekurencji | częściej, bo większość realnej pracy dzieje się na tabelach |

**Ważna, często pomijana subtelność:** `Table.Buffer` **nie gwarantuje** w 100% przypadków pełnego wymuszenia obliczenia *każdej* kolumny, jeśli dana kolumna nigdy nie jest faktycznie użyta w dalszych krokach (silnik M bywa "leniwy kolumnowo" nawet po buforowaniu). Praktyczna zasada: **jeśli chcesz mieć pewność zamrożenia konkretnej kolumny (np. losowej, czasowej), odwołaj się do niej explicite** albo bufferuj wynik `Table.SelectColumns` ograniczony tylko do potrzebnych kolumn — to też **redukuje zużycie pamięci**.


## 6. Klasyczna pułapka wydajnościowa: `List.Generate` bez buforowania

To jest **najbardziej znany, "podręcznikowy" przypadek użycia** `List.Buffer` w całej społeczności Power Query (opisywany m.in. przez Chrisa Webba i Bena Gribaudo). Zrozumienie go daje Ci pełne zrozumienie *dlaczego* Buffer działa.

### Scenariusz: liczymy skumulowaną sumę (running total) krok po kroku, symulując pętlę

**Wersja BEZ buforowania — wygląda niewinnie, ale jest wykładniczo wolna:**


In [ ]:
// ANTY-PATTERN — NIE RÓB TEGO na dużych danych
let
    Zrodlo = {1..2000},  // 2000 liczb
    // Budujemy listę skumulowanych sum przez List.Generate,
    // gdzie każdy kolejny element odwołuje się do CAŁEJ historii poprzednich wyników
    SkumulowanaSuma = List.Generate(
        () => [i = 0, suma = 0],
        each [i] < List.Count(Zrodlo),
        each [i = [i] + 1, suma = [suma] + Zrodlo{[i]}],
        each [suma]
    )
in
    SkumulowanaSuma

Dlaczego to jest wolne? Bo `Zrodlo{[i]}` wewnątrz `List.Generate` **odwołuje się do `Zrodlo` w każdej z 2000 iteracji**. Jeśli `Zrodlo` nie jest zbuforowane, a jest wynikiem jakiegoś kroku wcześniej (zwłaszcza jeśli source to zapytanie do bazy danych albo skomplikowana transformacja), to **silnik M może próbować ponownie ewaluować tę zależność przy każdym dostępie**, a przy operacjach *self-referencing* (typu running total, gdzie kolejny krok zależy od poprzedniego) obserwowany jest efekt **kwadratowego lub gorszego wzrostu czasu wykonania** wraz z liczbą wierszy — 2000 wierszy może trwać sekundy, a 50 000 może się nie doczekać w ogóle.

### Wersja Z buforowaniem — ten sam wynik, drastycznie szybciej:


In [ ]:
// POPRAWNY WZORZEC — bufferujemy źródło PRZED użyciem w pętli generującej
let
    Zrodlo = List.Buffer({1..2000}),  // <- kluczowa zmiana: jedna linijka
    SkumulowanaSuma = List.Generate(
        () => [i = 0, suma = 0],
        each [i] < List.Count(Zrodlo),
        each [i = [i] + 1, suma = [suma] + Zrodlo{[i]}],
        each [suma]
    )
in
    SkumulowanaSuma

**Efekt w praktyce:** ten sam kod, dla kilkudziesięciu tysięcy wierszy, potrafi przyspieszyć z "kilku minut / zawieszenia" do **poniżej sekundy**. To nie jest przesada — to jest jeden z najczęściej cytowanych case'ów wydajnościowych w Power Query w ogóle.

### Reguła praktyczna
> **Każda lista/tabela, do której odwołujesz się więcej niż raz wewnątrz `List.Generate`, `List.Accumulate` albo dowolnej logiki iteracyjnej/rekurencyjnej — powinna być owinięta w `List.Buffer` (lub `Table.Buffer`, jeśli to tabela) ZANIM wejdzie w pętlę.**


## 7. Zaawansowane triki produkcyjne

### 7.1 Stabilizacja danych "niestabilnych" (volatile sources)

Jeśli kolumna zawiera coś, co **zmienia się przy każdym odwołaniu** — np. `DateTime.LocalNow()`, `Number.Random()`, dane z żywego API — to bez buforowania **różne kroki zapytania mogą zobaczyć różne wartości** tej samej "logicznej" kolumny, bo silnik przelicza ją od nowa przy każdym dostępie.


In [ ]:
let
    Zrodlo = Table.FromRecords({[a=1],[a=2],[a=3]}),
    // BEZ buforowania: Losowa może dać RÓŻNE wartości
    // w kolumnie "Losowa" i w kolumnie "LosowaZaokraglona",
    // mimo że logicznie powinny pochodzić z tej samej wartości
    DodajLosowa = Table.AddColumn(Zrodlo, "Losowa", each Number.Random()),
    Zbuforowane = Table.Buffer(DodajLosowa),  // <- zamrażamy wylosowane wartości TERAZ
    DodajZaokraglona = Table.AddColumn(Zbuforowane, "LosowaZaokraglona", each Number.Round([Losowa], 2))
in
    DodajZaokraglona

**Zasada:** wszędzie, gdzie w zapytaniu jest jakikolwiek element losowości, czasu "na żywo" albo odczytu z API, który może zwrócić inny wynik przy drugim wywołaniu — **buforuj zaraz po jego wygenerowaniu**, żeby "zamrozić" wartość na resztę zapytania.

### 7.2 Self-joins i wielokrotne odwołania do tej samej tabeli

Gdy robisz `Table.NestedJoin` / merge tabeli **samej ze sobą** albo używasz tej samej tabeli jako referencji w kilku różnych miejscach (np. lookup w `each`), bez buforowania silnik może **wielokrotnie odpytywać źródło** (np. SQL Server) — raz na każde odwołanie.


In [ ]:
let
    Zamowienia = Sql.Database("Serwer", "Baza"){[Schema="dbo", Item="Zamowienia"]}[Data],
    // Buforujemy RAZ, bo poniżej używamy tej tabeli 2x: w merge i w add column
    ZamowieniaBuf = Table.Buffer(Zamowienia),

    Polaczone = Table.NestedJoin(
        ZamowieniaBuf, {"KlientID"},
        ZamowieniaBuf, {"KlientID"},   // self-join na tej samej, zbuforowanej kopii
        "PoprzednieZamowienia", JoinKind.LeftOuter
    ),
    ZLicznikiem = Table.AddColumn(Polaczone, "LiczbaZamowienKlienta", each
        Table.RowCount(Table.SelectRows(ZamowieniaBuf, (w) => w[KlientID] = [KlientID]))
    )
in
    ZLicznikiem

### 7.3 Ranking / running total per wiersz — buforuj TYLKO potrzebną kolumnę

Klasyczny błąd: buforowanie **całej tabeli** (ze wszystkimi kolumnami), gdy w logice `each` potrzebujesz tylko jednej kolumny do porównań. To marnuje pamięć i czas.


In [ ]:
let
    Zrodlo = Table.Buffer(TabelaWejsciowa),

    // GORZEJ: bufferujesz 30 kolumn, a potrzebujesz jednej
    // ZaSlabo = Table.Buffer(Zrodlo)

    // LEPIEJ: buforuj tylko kolumnę, która faktycznie wchodzi w powtarzalne porównania
    TylkoIlosci = List.Buffer(Table.Column(Zrodlo, "Ilosc")),

    ZRankingiem = Table.AddColumn(Zrodlo, "Ranking", each
        List.PositionOf(List.Sort(TylkoIlosci, Order.Descending), [Ilosc]) + 1
    )
in
    ZRankingiem

**Reguła:** buforuj **najwęższy możliwy zakres danych** (pojedynczą kolumnę jako listę, a nie całą tabelę), jeśli tylko ta kolumna jest potrzebna w powtarzalnym dostępie. Mniej danych w RAM = szybciej i bezpieczniej dla pamięci.

### 7.4 Buforowanie wyników pośrednich w długich łańcuchach transformacji

Jeśli masz zapytanie z 20+ krokami, gdzie krok 15 jest bardzo kosztowny (np. duży `Table.Group` albo `Table.AddColumn` z ciężką logiką), a kroki 16–20 **wielokrotnie odwołują się do wyniku kroku 15** (a nie tylko liniowo, krok po kroku) — zbuforuj wynik kroku 15 raz, zanim rozgałęzisz logikę.

### 7.5 Kombinacja z `Table.RemoveColumns` / `Table.SelectColumns` przed buforowaniem

Zawsze, kiedy to możliwe: **najpierw ogranicz kolumny i wiersze do minimum, a dopiero potem buforuj**. Kolejność ma znaczenie dla ilości danych faktycznie trafiających do RAM:


In [ ]:
// LEPSZA kolejność: filtruj i wybieraj kolumny PRZED buforowaniem
let
    Zrodlo = TabelaZrodlowa,
    Filtr = Table.SelectRows(Zrodlo, each [Status] = "Aktywny"),
    WybraneKolumny = Table.SelectColumns(Filtr, {"ID", "Ilosc", "KlientID"}),
    Zbuforowane = Table.Buffer(WybraneKolumny)  // buforujemy już "chudą" tabelę
in
    Zbuforowane

## 8. Buffer a query folding — najważniejszy kompromis w całym temacie

To jest **najczęściej pomijana konsekwencja** stosowania Buffer, a jednocześnie najważniejsza dla Ciebie jako osoby pracującej głównie z SQL Server.

### Co to jest query folding (przypomnienie)

Query folding to mechanizm, w którym Power Query **tłumaczy Twoje kroki M z powrotem na natywne zapytanie źródła** (np. SQL) i wykonuje filtrowanie/agregację/join **po stronie serwera**, zamiast ściągać surowe dane i przetwarzać je lokalnie w silniku M.

### Dlaczego Buffer to przerywa

`List.Buffer` / `Table.Buffer` **wymuszają pełną materializację danych w pamięci lokalnej w tym konkretnym momencie**. To fizycznie oznacza: *"ściągnij dane z serwera TERAZ, w całości"*. Po takim kroku **nie ma już czego foldować** — wszystkie kolejne transformacje (filtry, agregacje, joiny) są wykonywane **lokalnie, w silniku M, na Twoim komputerze**, a nie na serwerze SQL.

### Konsekwencja praktyczna

| Sytuacja | Skutek |
|---|---|
| Buforujesz **na początku** zapytania, przed filtrami | Tracisz folding dla WSZYSTKICH kolejnych kroków — nawet prostego `Table.SelectRows`, który serwer zrobiłby w milisekundy, teraz robi Twój laptop na całej tabeli |
| Buforujesz **na końcu** zapytania, po tym jak folding i tak by się urwał (np. po `Table.AddColumn` z logiką `each`, co zwykle *już* przerywa folding) | Brak dodatkowej straty — folding był i tak przerwany wcześniej |
| Źródło **nie foldowalne z natury** (plik Excel, CSV, API, folder) | Buffer nie "kosztuje" utraty foldingu, bo go tam nigdy nie było |

### Złota zasada kolejności kroków

> **Rób wszystko, co może się foldować (filtry, `Table.SelectRows`, `Table.Group`, proste joiny) NAJPIERW — jak najbliżej źródła. Buforuj JAK NAJPÓŹNIEJ, dopiero tuż przed logiką, która i tak foldingu nie obsługuje (pętle, `each` z odwołaniami do innych tabel, `List.Generate`).**

### Jak sprawdzić, czy krok się foldował

Kliknij prawym przyciskiem na krok w okienku "Applied Steps" → jeśli opcja **"View Native Query"** jest aktywna (nie wyszarzona) — ten krok wciąż się folduje. W momencie, gdy ta opcja znika (jest wyszarzona) na jakimś kroku — folding urwał się o krok wcześniej. To najszybszy sposób, żeby zdecydować, **gdzie dokładnie w łańcuchu kroków warto wstawić `Table.Buffer`**, żeby nie zepsuć foldingu, który i tak by się przydał.


## 9. Kiedy **NIE** stosować Buffer — checklist

Buffer to nie jest "przycisk przyspiesz", którego dokładasz wszędzie "na wszelki wypadek". Źle użyty **spowalnia** zapytanie i zwiększa ryzyko błędów pamięci. Nie stosuj, gdy:

1. **Lista/tabela jest referencjonowana tylko raz.** Buforowanie czegoś, do czego i tak odwołujesz się jednorazowo, to czysty narzut (koszt materializacji) bez żadnej korzyści.
2. **Dane są bardzo duże, a Twoje zapytanie i tak by się foldowało do SQL.** Wtedy serwer SQL (z indeksami, statystykami, optymalizatorem) niemal zawsze wygra z lokalnym przetwarzaniem tabeli wielu milionów wierszy w pamięci Twojego laptopa. Buforowanie takiej tabeli w całości może doprowadzić do **wyczerpania pamięci RAM** (`Out of memory`) albo drastycznego spowolnienia przez swapowanie na dysk.
3. **Pracujesz w trybie DirectQuery w Power BI.** Tam nie ładujesz danych do modelu — buforowanie w Power Query nie ma sensu koncepcyjnego w tym kontekście (a w niektórych przypadkach bywa wręcz niedozwolone/ignorowane).
4. **Chcesz jeszcze dalej filtrować/agregować dane, a ten etap mógłby się foldować.** Buforowanie za wcześnie zabija folding dla operacji, które serwer zrobiłby export razy szybciej.
5. **Źródło jest już małe i tanie w odczycie** (np. mały plik konfiguracyjny, tabela wymiarów z kilkoma wierszami). Zysk z buforowania jest wtedy zerowy lub ujemny (sam narzut wywołania funkcji).
6. **Dodajesz Buffer "profilaktycznie", bez zmierzenia realnego problemu.** To antywzorzec — buforowanie **zawsze** kosztuje pamięć i czas na materializację. Stosuj je jako **lek na konkretny, zaobserwowany objaw** (wolne odświeżanie, powtarzające się zapytania do źródła w logach SQL Profiler), nie jako rutynę "dla bezpieczeństwa".

### Sygnały ostrzegawcze, że przesadzasz z Buffer
- Zapytanie zużywa dużo więcej pamięci niż wynikałoby z rozmiaru danych na wyjściu.
- Odświeżanie stało się **wolniejsze** po dodaniu Buffer (klasyczny znak: zepsułeś folding, a lokalne przetwarzanie jest wolniejsze niż serwerowe).
- Buforujesz tabelę/listę, a bezpośrednio pod spodem w kodzie ta zmienna jest użyta dokładnie raz.


## 10. Jak zmierzyć efekt buforowania (Query Diagnostics)

Nie zgaduj — mierz. Power Query ma wbudowane narzędzie **Query Diagnostics** (zakładka *Tools* w Power Query Editor):

1. **Start Diagnostics** → odśwież/wykonaj zapytanie → **Stop Diagnostics**.
2. Power Query wygeneruje kilka tabel diagnostycznych (`Detailed`, `Aggregated` itd.) pokazujących: czas trwania każdego kroku, liczbę wywołań danego kroku (kolumna typu `Exclusive Duration`, `Id`, `Operation`) i — kluczowe — **czy i ile razy dany krok został przeliczony**.
3. **Metoda porównawcza:** uruchom diagnostykę na wersji BEZ `Buffer`, zapisz wyniki. Dodaj `Buffer` w podejrzanym miejscu, uruchom diagnostykę ponownie. Porównaj:
   - sumaryczny czas odświeżania (`Aggregated` → suma `Exclusive Duration`),
   - liczbę odwołań do źródła danych (w logu zobaczysz powtarzające się zapytania SQL, jeśli źródło nie jest zbuforowane — to najbardziej jednoznaczny sygnał problemu).
4. Dla źródeł SQL Server możesz dodatkowo uruchomić **SQL Server Profiler / Extended Events** równolegle z odświeżaniem w Power BI — zobaczysz "gołym okiem", czy to samo zapytanie SQL jest wysyłane 1 raz, czy N razy. To najbardziej przekonujący dowód na to, że Buffer faktycznie coś naprawił (albo że wcale nie był potrzebny).

### Prosty "manualny" test A/B bez narzędzi diagnostycznych
Zmierz czas odświeżania stoperem (albo `#duration` na starcie/końcu w osobnym kroku diagnostycznym) dla wersji z i bez Buffer, na realnym wolumenie danych (nie na próbce 10 wierszy — tam różnice się nie ujawnią).


## 11. Cheatsheet / ściąga decyzyjna

| Pytanie | Odpowiedź |
|---|---|
| Odwołuję się do listy/tabeli **więcej niż raz**? | Jeśli tak → rozważ Buffer |
| Jestem wewnątrz `List.Generate` / logiki rekurencyjnej / running total? | **Prawie zawsze** buforuj źródło iteracji |
| Mam kolumnę z losowością / czasem "na żywo" / danymi z API, które mogą się zmienić między odczytami? | Buforuj **zaraz po** jej wygenerowaniu |
| Robię self-join albo wielokrotny lookup do tej samej tabeli? | Buforuj tabelę **raz**, przed pierwszym użyciem |
| Moje zapytanie foldowałoby się do SQL, gdybym nie przerwał tego Bufferem? | Przesuń Buffer **jak najpóźniej**, po wszystkich krokach foldowalnych |
| Dane są bardzo duże (miliony wierszy) i mieszczą się w zasięgu foldingu? | **Nie buforuj** — zostaw robotę serwerowi |
| Referencjonuję to tylko raz? | **Nie buforuj** — to czysty narzut |
| Potrzebuję tylko jednej kolumny w powtarzalnym porównaniu? | Buforuj `Table.Column(...)` jako `List.Buffer`, nie całą tabelę |
| Nie zmierzyłem jeszcze problemu, tylko "na wszelki wypadek" dodaję Buffer? | **Zatrzymaj się** — zmierz najpierw (sekcja 10) |


## 12. Ćwiczenia do samodzielnego przetestowania

Żeby to naprawdę "wskoczyło" — polecam samodzielnie odtworzyć te eksperymenty na swoim komputerze (Power BI Desktop, pusta tabela testowa):

1. **Eksperyment podstawowy:** odtwórz przykład z sekcji 6 (`List.Generate` running total) dla `{1..500}`, `{1..5000}` i `{1..20000}` — z Bufferem i bez. Zmierz czas stoperem. Zaobserwuj nieliniowy wzrost czasu w wersji bez Buffer.
2. **Eksperyment z query folding:** weź tabelę z SQL Server, dodaj `Table.Buffer` na samym początku zapytania (zaraz po `Source`), a potem dodaj filtr `Table.SelectRows`. Sprawdź prawym przyciskiem na kroku filtra, czy "View Native Query" jest dostępne (**nie powinno być** — bo Buffer je zabił). Usuń Buffer, sprawdź ponownie (**powinno być** dostępne).
3. **Eksperyment z losowością:** stwórz tabelę z kolumną `Number.Random()`, dodaj kolejną kolumnę bazującą na tej pierwszej — raz bez Buffer (zobacz, że wartości "nie zgadzają się" logicznie), raz z Buffer w środku (zobacz spójność).
4. **Eksperyment z Query Diagnostics:** uruchom diagnostykę na zapytaniu z self-joinem tabeli SQL do samej siebie — raz bez buforowania (zobacz powtarzające się zapytania do źródła w logu), raz z `Table.Buffer` (zobacz, że zapytanie do źródła pojawia się tylko raz).

Po przejściu tych czterech eksperymentów będziesz mieć nie tylko wiedzę teoretyczną, ale **namacalne, zmierzone przez Ciebie samego dowody**, kiedy Buffer pomaga, a kiedy szkodzi — a to jest jedyny naprawdę trwały sposób nauki tego tematu.
